In [153]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')

### Load the Data

In [154]:
#relative path
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
data_path = os.path.join(parent_dir, 'Data', 'train.csv')
test_path = os.path.join(parent_dir, 'Data', 'test.csv')

# Load data and test data
data = pd.read_csv(data_path, index_col = 0)
test = pd.read_csv(test_path, index_col = 0)

# Separate data into X and Y
y = data.SalePrice
X = data.drop("SalePrice", axis = 1)

In [155]:
num_var = X.select_dtypes(include=[np.number]).columns
cat_var = X.select_dtypes(include=[np.object_]).columns
print(num_var.shape[0], "numerical variables")
print(cat_var.shape[0], "categorical variables")
print(X.shape[1], "total features")

36 numerical variables
43 categorical variables
79 total features


### All variables
Filled missing categorical with the mode of that feature (for categorical)

In [156]:
def mean_median(feat:str, args, index):
    '''
    input
        feat: feature name
        args: feat value name in the dataset
        index: row index name
    '''
    assert len(index) == len(args), "length of index and unique feat value is different"

    df = pd.DataFrame({"Mean": [0]*len(index), "Median": [0]*len(args), "n": [0]*len(args)}, index = index)
    for ind, arg in enumerate(args):
        if type(arg) is not list:
            df.iloc[ind, 0] = data[data[feat] == arg].SalePrice.mean()
            df.iloc[ind, 1] = data[data[feat] == arg].SalePrice.median()
            df.iloc[ind, 2] = len(data[data[feat] == arg])
        else:
            df.iloc[ind, 0] = data[data[feat].isin(arg)].SalePrice.mean()
            df.iloc[ind, 1] = data[data[feat].isin(arg)].SalePrice.median()
            df.iloc[ind, 2] = len(data[data[feat].isin(arg)])
    display(df.round(2))
    return data[feat].unique()

def fill_na_inplace(feats:list):
    print("Missing in test but not in train data (if any):")
    for feat in feats:
        if X[feat].isnull().sum() > 0:
            X[feat] = X[feat].fillna("NA")
            test[feat] = test[feat].fillna("NA")
        elif test[feat].isnull().sum() > 0:
            print(feat+":", test[feat].isnull().sum())
            test[feat] = test[feat].fillna(test[feat].mode()[0])

def check_numtype(feats:list):
    for feat in feats:
        if X[feat].nunique() < test[feat].nunique():
            print(feat, "in test set has unseen feature in training data")
        else:
            print(feat, ": All values of under test data has appear in the training data")

In [157]:
fill_na_inplace(ohe_var+ord_var)
print()
check_numtype(ohe_var)

Missing in test but not in train data (if any):

BldgType : All values of under test data has appear in the training data
CentralAir : All values of under test data has appear in the training data
Condition1 : All values of under test data has appear in the training data
Condition2 : All values of under test data has appear in the training data
Foundation : All values of under test data has appear in the training data
Heating : All values of under test data has appear in the training data
HouseStyle : All values of under test data has appear in the training data
LandContour : All values of under test data has appear in the training data
LotConfig : All values of under test data has appear in the training data
Neighborhood : All values of under test data has appear in the training data
RoofMatl : All values of under test data has appear in the training data
RoofStyle : All values of under test data has appear in the training data
SaleCondition : All values of under test data has appear 

### Numerical Variables

In [158]:
numerical_missing = data[num_var].isnull().sum(axis = 0)
display(numerical_missing[numerical_missing>0])
numerical_missing_test = test[num_var].isnull().sum(axis = 0)
display(numerical_missing_test[numerical_missing_test>0])

LotFrontage    259
MasVnrArea       8
GarageYrBlt     81
dtype: int64

LotFrontage     227
MasVnrArea       15
BsmtFinSF1        1
BsmtFinSF2        1
BsmtUnfSF         1
TotalBsmtSF       1
BsmtFullBath      2
BsmtHalfBath      2
GarageYrBlt      78
GarageCars        1
GarageArea        1
dtype: int64

In [159]:
data.LotFrontage.describe()

count    1201.000000
mean       70.049958
std        24.284752
min        21.000000
25%        59.000000
50%        69.000000
75%        80.000000
max       313.000000
Name: LotFrontage, dtype: float64

### Categorical Variables

In [160]:
# One Hot Encoding Variables (43 categorical 29+14)
ohe_var = ["BldgType", "CentralAir", "Condition1", "Condition2", "Foundation", 
                                             "Heating", "HouseStyle", "LandContour", "LotConfig", "Neighborhood", 
                                             "RoofMatl", "RoofStyle", "SaleCondition", "Street",
                                             "Alley", 'Electrical', 'Fence', 'FireplaceQu', 'GarageCond', 'GarageType', 'MasVnrType',
                                             'MiscFeature', 'PoolQC', 'Exterior1st', 'Exterior2nd', 'Functional', 'MSZoning', 'SaleType', 'Utilities']
ord_var = ["ExterCond", "ExterQual", "HeatingQC", "LandSlope", "LotShape", "PavedDrive",
                    'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'BsmtQual', 'GarageFinish', 'GarageQual',
                    'KitchenQual'] 
assert len(ohe_var + ord_var) == 43, "Didn't include all Categorical Variables"

#### One Hot Encoding

In [161]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(drop="first", sparse_output=False) # Handle unknown by default is error

# One Hot Encoding Matrix
ohe_matrix = pd.DataFrame(ohe.fit_transform(X[ohe_var]))
ohe_matrix_test = pd.DataFrame(ohe.transform(test[ohe_var])) # fit in the training data so that test data has the same dimension

# Put index and column name back
ohe_matrix.index = X.index
ohe_matrix_test.index = test.index

ohe_columns = ohe.get_feature_names_out(ohe_var).astype(str)
ohe_matrix.columns = ohe_columns
ohe_matrix_test.columns = ohe_columns

# Concat back to "data" and test
data = pd.concat([data, ohe_matrix], axis = 1)
test = pd.concat([test, ohe_matrix_test], axis = 1)

ValueError: Found unknown categories [nan] in column 23 during transform

#### Ordinal Encoding

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# ord_var = ["ExterCond", "ExterQual", "HeatingQC", "LandSlope", "LotShape", "PavedDrive",
#                     'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'BsmtQual', 'GarageFinish', 'GarageQual',
#                     'KitchenQual'] 

categories = [['Po', 'Fa', 'TA', 'Gd', 'Ex'], # ExterCond
              ['Po', 'Fa', 'TA', 'Gd', 'Ex'], # ExterQual 
              ['Po', 'Fa', 'TA', 'Gd', 'Ex'], # HeatingQC
              ["Sev", "Mod", "Gtl"], # LandSlope
              ["IR3", "IR2", "IR1", "Reg"], # LotShape
              ["N", "P", "Y"], # PavedDrive
              ["NA", 'Po', 'Fa', 'TA', 'Gd', 'Ex'], # BsmtCond
              ['NA', 'No', 'Mn', 'Av', 'Gd'], # BsmtExposure
              ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', "GLQ"], # BsmtFinType1
              ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', "GLQ"], # BsmtFinType2
              ["NA", 'Po', 'Fa', 'TA', 'Gd', 'Ex'], # BsmtQual
              ["NA", "Unf", "RFn", "Fin"], # GarageFinish
              ["Ex", 'Gd', 'TA', 'Fa', 'Po', 'NA'], # GarageQual
              ['Ex', 'Gd', 'TA', 'Fa'] # KitchenQual
              ]

ord_encoder = OrdinalEncoder(categories=categories)

# Ordinal Encoding Matrix
ord_matrix = pd.DataFrame(ord_encoder.fit_transform(X[ord_var]))
ord_matrix_test = pd.DataFrame(ord_encoder.transform(test[ord_var]))

# Put index and column name back
ord_matrix.index = X.index
ord_matrix_test.index = test.index

ord_matrix.columns = ord_var
ord_matrix_test.columns = ord_var

# replace "data" and test with ordinal features
data[ord_var] = ord_matrix
test[ord_var] = ord_matrix_test

### Save Clean Data

In [ ]:
# path
clean_data_path = os.path.join(parent_dir, 'Data', 'train_clean.csv')
clean_test_path = os.path.join(parent_dir, 'Data', 'test_clean.csv')

# clean_features
clean_feat_test = list(num_var) + list(ohe_columns) + list(ord_var)
clean_feat = clean_feat_test + [y.name]
len(clean_feat)

# save final data to the path
data[clean_feat].to_csv(clean_data_path)
test[clean_feat_test].to_csv(clean_test_path)